In [1]:
# -*- coding: utf-8 -*-
"""
Colab-ready end-to-end machine learning pipeline for stroke prediction.

Purpose
-------
1) Handle missing BMI values using multivariable imputation.
2) Address severe class imbalance in stroke prediction.
3) Compare multiple machine learning models with 5-fold stratified cross-validation.
4) Prioritize sensitivity >= 0.80, then maximize ROC-AUC.
5) Generate ROC, PR, calibration plots, and SHAP explainability outputs.

Input
-----
- CSV file with the stroke dataset.
  Expected columns:
    id, gender, age, hypertension, heart_disease, ever_married,
    work_type, Residence_type, avg_glucose_level, bmi,
    smoking_status, stroke

Output
------
- Model comparison CSV
- Threshold summary CSV
- ROC curve PNG
- PR curve PNG
- Calibration plot PNG
- SHAP summary plot PNG
- SHAP waterfall plot PNG
- SHAP top-10 feature CSV
- Final JSON summary

Notes
-----
- This script is designed for Google Colab.
- It uses out-of-fold probabilities from 5-fold stratified CV for evaluation.
- Thresholds are selected using:
    (a) Youden's J index
    (b) Sensitivity-constrained threshold (target sensitivity >= 0.80)
- Class imbalance handling methods compared:
    1. None/threshold optimization
    2. Class weight adjustment
    3. SMOTE
- Models compared:
    - Logistic Regression
    - Decision Tree
    - Random Forest
    - Extra Trees
    - XGBoost
    - MLP

Author guidance
---------------
- Set RANDOM_STATE for full reproducibility.
- Review final threshold before clinical deployment.
- External validation is strongly recommended.
"""

# =========================
# 0. Package installation
# =========================
# In Colab, uncomment the next line if needed.
# !pip -q install xgboost shap imbalanced-learn


# =========================
# 1. Imports and settings
# =========================
import os
import json
import random
import warnings
from dataclasses import dataclass
from typing import Dict, Tuple, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

import shap

RANDOM_STATE = 42
N_SPLITS = 5
TARGET_SENSITIVITY = 0.80
DPI = 300

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

plt.rcParams["figure.dpi"] = DPI
plt.rcParams["savefig.dpi"] = DPI
plt.rcParams["font.size"] = 10


# ============================================
# 2. Utility functions and dataclass container
# ============================================
@dataclass
class ModelResult:
    model_name: str
    imbalance_method: str
    threshold_rule: str
    threshold: float
    roc_auc: float
    pr_auc: float
    average_precision: float
    brier_score: float
    sensitivity: float
    specificity: float
    precision: float
    tn: int
    fp: int
    fn: int
    tp: int


def validate_input_dataframe(df: pd.DataFrame) -> None:
    """
    Validate required columns and target variable integrity.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataset.

    Raises
    ------
    ValueError
        If required columns are missing or target is invalid.
    """
    required_cols = [
        "id", "gender", "age", "hypertension", "heart_disease",
        "ever_married", "work_type", "Residence_type",
        "avg_glucose_level", "bmi", "smoking_status", "stroke"
    ]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    unique_target = sorted(df["stroke"].dropna().unique().tolist())
    if unique_target not in ([0, 1], [0], [1]):
        raise ValueError("Target column 'stroke' must be binary coded as 0/1.")



def compute_classification_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, Any]:
    """
    Compute binary classification performance metrics at a chosen threshold.

    Parameters
    ----------
    y_true : np.ndarray
        True binary labels.
    y_prob : np.ndarray
        Predicted probabilities for the positive class.
    threshold : float
        Probability threshold for converting probabilities to labels.

    Returns
    -------
    Dict[str, Any]
        Dictionary containing confusion matrix and summary metrics.
    """
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision),
    }



def get_youden_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """
    Determine optimal threshold using Youden's J index.

    Returns
    -------
    float
        Threshold maximizing TPR - FPR.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    return float(thresholds[best_idx])



def get_sensitivity_constrained_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    target_sensitivity: float = 0.80
) -> float:
    """
    Select the highest threshold that still achieves target sensitivity.

    Rationale
    ---------
    In screening-like clinical settings, maintaining sensitivity is prioritized.
    Using the highest threshold that still satisfies sensitivity preserves as much
    specificity as possible under the sensitivity constraint.

    Returns
    -------
    float
        Selected threshold.
    """
    thresholds = np.unique(y_prob)
    thresholds = np.sort(thresholds)

    chosen = thresholds[0]
    for thr in thresholds:
        metrics = compute_classification_metrics(y_true, y_prob, thr)
        if metrics["sensitivity"] >= target_sensitivity:
            chosen = thr
    return float(chosen)



def make_preprocessor(numeric_features: List[str], categorical_features: List[str]) -> ColumnTransformer:
    """
    Create preprocessing pipeline for numeric and categorical variables.

    Numeric pipeline:
        Iterative imputation + standardization
    Categorical pipeline:
        One-hot encoding
    """
    numeric_transformer = SkPipeline(
        steps=[
            ("imputer", IterativeImputer(random_state=RANDOM_STATE, sample_posterior=False, max_iter=20)),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = SkPipeline(
        steps=[
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )
    return preprocessor



def build_model_catalog(scale_pos_weight: float) -> Dict[str, Any]:
    """
    Define candidate machine learning models.

    Parameters
    ----------
    scale_pos_weight : float
        Ratio of negative to positive class used for XGBoost imbalance handling.

    Returns
    -------
    Dict[str, Any]
        Dictionary of initialized model objects.
    """
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        ),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=20,
            random_state=RANDOM_STATE,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=500,
            random_state=RANDOM_STATE,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=2,
            tree_method="hist",
        ),
    }



def apply_class_weight(model: Any) -> Any:
    """
    Return a cloned model with class weights adjusted if supported.
    """
    m = clone(model)
    params = m.get_params()

    if "class_weight" in params:
        m.set_params(class_weight="balanced")
    elif isinstance(m, XGBClassifier):
        # scale_pos_weight already handled at model initialization
        pass
    return m



def generate_oof_probabilities(
    X: pd.DataFrame,
    y: pd.Series,
    model: Any,
    imbalance_method: str,
    numeric_features: List[str],
    categorical_features: List[str],
    n_splits: int = 5,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate out-of-fold predicted probabilities using stratified CV.

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix.
    y : pd.Series
        Binary outcome.
    model : estimator
        Classifier implementing fit/predict_proba.
    imbalance_method : str
        One of {'none', 'class_weight', 'smote'}.
    numeric_features : list[str]
        Numeric column names.
    categorical_features : list[str]
        Categorical column names.
    n_splits : int
        Number of CV folds.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray]
        y_true, out-of-fold probabilities.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    y_true_all = []
    y_prob_all = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        preprocessor = make_preprocessor(numeric_features, categorical_features)

        try:
            if imbalance_method == "smote":
                pipe = ImbPipeline(
                    steps=[
                        ("preprocess", preprocessor),
                        ("smote", SMOTE(random_state=RANDOM_STATE)),
                        ("model", clone(model)),
                    ]
                )
            else:
                pipe = SkPipeline(
                    steps=[
                        ("preprocess", preprocessor),
                        ("model", clone(model)),
                    ]
                )

            pipe.fit(X_train, y_train)
            y_prob = pipe.predict_proba(X_valid)[:, 1]
        except Exception as e:
            raise RuntimeError(f"Fold {fold} failed for model={type(model).__name__}, imbalance={imbalance_method}: {e}")

        y_true_all.extend(y_valid.tolist())
        y_prob_all.extend(y_prob.tolist())

    return np.array(y_true_all), np.array(y_prob_all)



def plot_combined_roc(model_curves: Dict[str, Tuple[np.ndarray, np.ndarray, float]], save_path: str) -> None:
    """Create combined ROC plot for all candidate models."""
    plt.figure(figsize=(8, 6))
    for label, (fpr, tpr, auc_val) in model_curves.items():
        plt.plot(fpr, tpr, linewidth=2, label=f"{label} (AUC={auc_val:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("AUROC Curves Across All Candidate Models")
    plt.legend(loc="lower right", fontsize=8, frameon=False)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()



def plot_combined_pr(model_curves: Dict[str, Tuple[np.ndarray, np.ndarray, float]], prevalence: float, save_path: str) -> None:
    """Create combined Precision-Recall plot for all candidate models."""
    plt.figure(figsize=(8, 6))
    for label, (recall, precision, ap_val) in model_curves.items():
        plt.plot(recall, precision, linewidth=2, label=f"{label} (AP={ap_val:.3f})")
    plt.axhline(prevalence, linestyle="--", linewidth=1, label=f"Baseline prevalence={prevalence:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("AUPRC Curves Across All Candidate Models")
    plt.legend(loc="upper right", fontsize=8, frameon=False)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()



def plot_combined_calibration(y_true: np.ndarray, prob_dict: Dict[str, np.ndarray], save_path: str) -> None:
    """Create combined calibration plot for all candidate models."""
    plt.figure(figsize=(8, 6))
    for label, probs in prob_dict.items():
        frac_pos, mean_pred = calibration_curve(y_true, probs, n_bins=10, strategy="quantile")
        plt.plot(mean_pred, frac_pos, marker="o", linewidth=1.5, label=label)
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed event rate")
    plt.title("Calibration Plot Across All Candidate Models")
    plt.legend(loc="upper left", fontsize=8, frameon=False)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()



def fit_final_pipeline(
    X: pd.DataFrame,
    y: pd.Series,
    model: Any,
    imbalance_method: str,
    numeric_features: List[str],
    categorical_features: List[str],
):
    """
    Fit the final model pipeline on the full dataset.
    """
    preprocessor = make_preprocessor(numeric_features, categorical_features)

    if imbalance_method == "smote":
        final_pipe = ImbPipeline(
            steps=[
                ("preprocess", preprocessor),
                ("smote", SMOTE(random_state=RANDOM_STATE)),
                ("model", clone(model)),
            ]
        )
    else:
        final_pipe = SkPipeline(
            steps=[
                ("preprocess", preprocessor),
                ("model", clone(model)),
            ]
        )

    final_pipe.fit(X, y)
    return final_pipe



def get_feature_names_from_preprocessor(preprocessor: ColumnTransformer) -> List[str]:
    """
    Retrieve transformed feature names after preprocessing.
    """
    feature_names = []

    num_features = preprocessor.transformers_[0][2]
    feature_names.extend(list(num_features))

    cat_encoder = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    cat_cols = preprocessor.transformers_[1][2]
    cat_names = cat_encoder.get_feature_names_out(cat_cols)
    feature_names.extend(list(cat_names))

    return feature_names



def run_shap_analysis(final_pipeline, X: pd.DataFrame, save_dir: str) -> Tuple[pd.DataFrame, str, str]:
    """
    Run SHAP analysis for the fitted final model.

    Supports tree-based models and linear logistic regression robustly.
    Uses a random subset for efficiency if necessary.

    Returns
    -------
    Tuple[pd.DataFrame, str, str]
        top10 dataframe, summary plot path, waterfall plot path
    """
    os.makedirs(save_dir, exist_ok=True)

    preprocess = final_pipeline.named_steps["preprocess"]
    model = final_pipeline.named_steps["model"]

    X_trans = preprocess.transform(X)
    feature_names = get_feature_names_from_preprocessor(preprocess)

    # Convert sparse matrix if necessary for SHAP compatibility
    if hasattr(X_trans, "toarray"):
        X_trans_dense = X_trans.toarray()
    else:
        X_trans_dense = np.asarray(X_trans)

    # Limit background/sample size for computational stability
    if X_trans_dense.shape[0] > 1000:
        rng = np.random.default_rng(RANDOM_STATE)
        sample_idx = rng.choice(X_trans_dense.shape[0], size=1000, replace=False)
        X_sample = X_trans_dense[sample_idx]
    else:
        X_sample = X_trans_dense

    # Choose explainer
    if isinstance(model, (RandomForestClassifier, ExtraTreesClassifier, DecisionTreeClassifier, XGBClassifier)):
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)
        if isinstance(shap_values, list):
            shap_values_plot = shap_values[1]
        else:
            shap_values_plot = shap_values
    elif isinstance(model, LogisticRegression):
        explainer = shap.LinearExplainer(model, X_sample, feature_perturbation="interventional")
        shap_values_plot = explainer.shap_values(X_sample)
    else:
        background = shap.sample(X_sample, min(100, X_sample.shape[0]), random_state=RANDOM_STATE)
        explainer = shap.KernelExplainer(model.predict_proba, background)
        shap_values = explainer.shap_values(X_sample[:200], nsamples=100)
        if isinstance(shap_values, list):
            shap_values_plot = shap_values[1]
            X_sample = X_sample[:200]
        else:
            shap_values_plot = shap_values
            X_sample = X_sample[:200]

    # Summary importance
    mean_abs_shap = np.abs(shap_values_plot).mean(axis=0)
    top10_df = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).head(10)

    top10_path = os.path.join(save_dir, "shap_top10_features.csv")
    top10_df.to_csv(top10_path, index=False)

    # Summary plot
    summary_path = os.path.join(save_dir, "shap_summary_plot.png")
    plt.figure()
    shap.summary_plot(shap_values_plot, X_sample, feature_names=feature_names, show=False, max_display=20)
    plt.tight_layout()
    plt.savefig(summary_path, bbox_inches="tight", dpi=DPI)
    plt.close()

    # Waterfall plot for a representative high-risk instance
    waterfall_path = os.path.join(save_dir, "shap_waterfall_plot.png")
    row_idx = int(np.argmax(np.abs(shap_values_plot).sum(axis=1)))

    try:
        expected_value = explainer.expected_value
        if isinstance(expected_value, list):
            expected_value = expected_value[1]
        exp = shap.Explanation(
            values=shap_values_plot[row_idx],
            base_values=expected_value,
            data=X_sample[row_idx],
            feature_names=feature_names,
        )
        plt.figure()
        shap.plots.waterfall(exp, max_display=15, show=False)
        plt.tight_layout()
        plt.savefig(waterfall_path, bbox_inches="tight", dpi=DPI)
        plt.close()
    except Exception:
        # Fallback bar plot if waterfall rendering is not available
        local_df = pd.DataFrame({
            "feature": feature_names,
            "shap_value": shap_values_plot[row_idx],
        }).sort_values("shap_value", key=np.abs, ascending=False).head(15)
        plt.figure(figsize=(8, 6))
        plt.barh(local_df["feature"][::-1], local_df["shap_value"][::-1])
        plt.title("Approximate local SHAP explanation")
        plt.tight_layout()
        plt.savefig(waterfall_path, bbox_inches="tight", dpi=DPI)
        plt.close()

    return top10_df, summary_path, waterfall_path


# =========================
# 3. Load and clean data
# =========================
# Google Colab file-open upload support
# This block opens a file chooser so the user can upload the CSV directly.
# If you prefer Google Drive, comment this block out and set DATA_PATH manually.

try:
    from google.colab import files  # type: ignore
    print("Please choose the stroke CSV file from your local computer.")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("No file was uploaded. Please upload a CSV file.")
    DATA_PATH = list(uploaded.keys())[0]
    print(f"Uploaded file detected: {DATA_PATH}")
except Exception as e:
    print("Colab upload widget was not available or upload failed.")
    print("Falling back to manual path setting.")
    # Example fallback path:
    DATA_PATH = "/content/3.stroke.csv"
    print(f"Fallback DATA_PATH = {DATA_PATH}")
    print(f"Upload block message: {e}")

OUTPUT_DIR = "/content/stroke_ml_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    df = pd.read_csv(DATA_PATH)
except Exception as e:
    raise FileNotFoundError(
        f"Failed to read dataset from {DATA_PATH}. Please verify the uploaded file or path. Original error: {e}"
    )

validate_input_dataframe(df)

# Basic cleaning
# Ultra-rare categories can destabilize some models. Collapse a single rare 'Other' gender entry.
df = df.copy()
df["gender"] = df["gender"].replace({"Other": "Female"})

y = df["stroke"].astype(int)
X = df.drop(columns=["stroke", "id"], errors="ignore")

numeric_features = ["age", "avg_glucose_level", "bmi"]
categorical_features = [c for c in X.columns if c not in numeric_features]

# Class imbalance ratio for XGBoost
n_pos = int(y.sum())
n_neg = int((1 - y).sum())
scale_pos_weight = n_neg / max(n_pos, 1)

print("Dataset shape:", df.shape)
print("Positive class proportion:", y.mean())
print("Missing values:\n", df.isnull().sum())
print(f"scale_pos_weight for XGBoost: {scale_pos_weight:.3f}")


# ============================================
# 4. Model comparison across imbalance methods
# ============================================
base_models = build_model_catalog(scale_pos_weight=scale_pos_weight)
imbalance_methods = ["none", "class_weight", "smote"]

results: List[ModelResult] = []
roc_plot_dict: Dict[str, Tuple[np.ndarray, np.ndarray, float]] = {}
pr_plot_dict: Dict[str, Tuple[np.ndarray, np.ndarray, float]] = {}
calibration_prob_dict: Dict[str, np.ndarray] = {}

oof_storage = {}  # store OOF probs for later plots/threshold checks

for model_name, base_model in base_models.items():
    for imbalance_method in imbalance_methods:
        try:
            if imbalance_method == "class_weight":
                model = apply_class_weight(base_model)
            else:
                model = clone(base_model)

            y_true_oof, y_prob_oof = generate_oof_probabilities(
                X=X,
                y=y,
                model=model,
                imbalance_method=imbalance_method,
                numeric_features=numeric_features,
                categorical_features=categorical_features,
                n_splits=N_SPLITS,
            )

            # Curves and scalar metrics
            roc_auc = roc_auc_score(y_true_oof, y_prob_oof)
            pr_auc = average_precision_score(y_true_oof, y_prob_oof)
            brier = brier_score_loss(y_true_oof, y_prob_oof)
            fpr, tpr, _ = roc_curve(y_true_oof, y_prob_oof)
            precision_arr, recall_arr, _ = precision_recall_curve(y_true_oof, y_prob_oof)

            # Store only the clinically most relevant version per model for combined plots:
            # prioritize class_weight, then none, then smote if not already stored.
            plot_label = f"{model_name} [{imbalance_method}]"
            roc_plot_dict[plot_label] = (fpr, tpr, roc_auc)
            pr_plot_dict[plot_label] = (recall_arr, precision_arr, pr_auc)
            calibration_prob_dict[plot_label] = y_prob_oof
            oof_storage[(model_name, imbalance_method)] = (y_true_oof, y_prob_oof)

            # Thresholds
            thr_youden = get_youden_threshold(y_true_oof, y_prob_oof)
            thr_sens = get_sensitivity_constrained_threshold(
                y_true_oof,
                y_prob_oof,
                target_sensitivity=TARGET_SENSITIVITY,
            )

            for rule_name, threshold in [
                ("youden_j", thr_youden),
                ("sensitivity_target", thr_sens),
            ]:
                m = compute_classification_metrics(y_true_oof, y_prob_oof, threshold)
                results.append(
                    ModelResult(
                        model_name=model_name,
                        imbalance_method=imbalance_method,
                        threshold_rule=rule_name,
                        threshold=threshold,
                        roc_auc=roc_auc,
                        pr_auc=pr_auc,
                        average_precision=pr_auc,
                        brier_score=brier,
                        sensitivity=m["sensitivity"],
                        specificity=m["specificity"],
                        precision=m["precision"],
                        tn=m["tn"],
                        fp=m["fp"],
                        fn=m["fn"],
                        tp=m["tp"],
                    )
                )

            print(f"Completed: {model_name} | {imbalance_method}")

        except Exception as e:
            print(f"Skipped: {model_name} | {imbalance_method} | Reason: {e}")

results_df = pd.DataFrame([r.__dict__ for r in results])
results_csv = os.path.join(OUTPUT_DIR, "model_comparison_full.csv")
results_df.to_csv(results_csv, index=False)

print("\nSaved:", results_csv)
print(results_df.head())


# ============================================
# 5. Select the optimal final model
# ============================================
# Clinical rule:
# 1) Sensitivity >= target
# 2) Among eligible models, maximize ROC-AUC
# 3) Tie-breakers: higher PR-AUC, lower Brier score
eligible = results_df[
    (results_df["threshold_rule"] == "sensitivity_target") &
    (results_df["sensitivity"] >= TARGET_SENSITIVITY)
].copy()

if eligible.empty:
    print("No model achieved the target sensitivity. Falling back to Youden-J ranking.")
    eligible = results_df[results_df["threshold_rule"] == "youden_j"].copy()

eligible = eligible.sort_values(
    by=["roc_auc", "pr_auc", "brier_score"],
    ascending=[False, False, True]
).reset_index(drop=True)

best_row = eligible.iloc[0]
print("\nBest model based on study criteria:")
print(best_row)

best_model_name = best_row["model_name"]
best_imbalance_method = best_row["imbalance_method"]
best_threshold_rule = best_row["threshold_rule"]
best_threshold = float(best_row["threshold"])

best_model = build_model_catalog(scale_pos_weight=scale_pos_weight)[best_model_name]
if best_imbalance_method == "class_weight":
    best_model = apply_class_weight(best_model)

summary_json = os.path.join(OUTPUT_DIR, "best_model_summary.json")
with open(summary_json, "w") as f:
    json.dump(best_row.to_dict(), f, indent=2)

print("Saved:", summary_json)


# ============================================
# 6. Combined AUROC/AUPRC/calibration plots
# ============================================
roc_path = os.path.join(OUTPUT_DIR, "all_models_roc_curve.png")
pr_path = os.path.join(OUTPUT_DIR, "all_models_pr_curve.png")
cal_path = os.path.join(OUTPUT_DIR, "all_models_calibration_plot.png")

plot_combined_roc(roc_plot_dict, roc_path)
plot_combined_pr(pr_plot_dict, prevalence=y.mean(), save_path=pr_path)
# Use one consistent y_true from any stored OOF output because all contain the same labels order from CV aggregation.
any_key = next(iter(oof_storage.keys()))
y_true_reference = oof_storage[any_key][0]
plot_combined_calibration(y_true_reference, calibration_prob_dict, cal_path)

print("Saved:", roc_path)
print("Saved:", pr_path)
print("Saved:", cal_path)


# ============================================
# 7. Fit final model on full dataset
# ============================================
final_pipeline = fit_final_pipeline(
    X=X,
    y=y,
    model=best_model,
    imbalance_method=best_imbalance_method,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
)

# Save fitted probabilities on full data for optional inspection
full_probs = final_pipeline.predict_proba(X)[:, 1]
full_pred = (full_probs >= best_threshold).astype(int)
final_pred_df = pd.DataFrame({
    "predicted_probability": full_probs,
    "predicted_label": full_pred,
    "observed_label": y.values,
})
preds_csv = os.path.join(OUTPUT_DIR, "final_model_predictions_full_data.csv")
final_pred_df.to_csv(preds_csv, index=False)
print("Saved:", preds_csv)


# ============================================
# 8. SHAP explainability for the final model
# ============================================
try:
    top10_df, shap_summary_path, shap_waterfall_path = run_shap_analysis(
        final_pipeline=final_pipeline,
        X=X,
        save_dir=OUTPUT_DIR,
    )
    print("Saved:", os.path.join(OUTPUT_DIR, "shap_top10_features.csv"))
    print("Saved:", shap_summary_path)
    print("Saved:", shap_waterfall_path)
    print("\nTop 10 SHAP features:")
    print(top10_df)
except Exception as e:
    print(f"SHAP analysis failed: {e}")


# ============================================
# 9. Display concise final summary
# ============================================
print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"Best model: {best_model_name}")
print(f"Imbalance method: {best_imbalance_method}")
print(f"Threshold rule: {best_threshold_rule}")
print(f"Threshold: {best_threshold:.4f}")
print(f"ROC-AUC: {best_row['roc_auc']:.3f}")
print(f"PR-AUC: {best_row['pr_auc']:.3f}")
print(f"Brier score: {best_row['brier_score']:.4f}")
print(f"Sensitivity: {best_row['sensitivity']:.3f}")
print(f"Specificity: {best_row['specificity']:.3f}")
print(f"Precision: {best_row['precision']:.3f}")
print(f"Confusion matrix: TN={int(best_row['tn'])}, FP={int(best_row['fp'])}, FN={int(best_row['fn'])}, TP={int(best_row['tp'])}")
print("Output directory:", OUTPUT_DIR)


# ============================================
# 10. EDA and Table 1 generation for baseline characteristics
# ============================================
def summarize_continuous(series: pd.Series) -> Tuple[str, str]:
    """
    Summarize a continuous variable as mean ± SD and median [IQR].

    Parameters
    ----------
    series : pd.Series
        Continuous variable.

    Returns
    -------
    Tuple[str, str]
        Formatted mean±SD string and median[IQR] string.
    """
    s = pd.to_numeric(series, errors="coerce").dropna()
    mean_sd = f"{s.mean():.2f} ± {s.std(ddof=1):.2f}"
    median_iqr = f"{s.median():.2f} [{s.quantile(0.25):.2f}–{s.quantile(0.75):.2f}]"
    return mean_sd, median_iqr


def create_missingness_table(df_input: pd.DataFrame, output_dir: str) -> pd.DataFrame:
    """
    Create a missingness summary table.

    Parameters
    ----------
    df_input : pd.DataFrame
        Input dataframe.
    output_dir : str
        Output directory.

    Returns
    -------
    pd.DataFrame
        Missingness summary.
    """
    missing_df = pd.DataFrame({
        "variable": df_input.columns,
        "n_missing": df_input.isna().sum().values,
        "pct_missing": (df_input.isna().mean().values * 100).round(2),
        "dtype": df_input.dtypes.astype(str).values,
    }).sort_values(["pct_missing", "variable"], ascending=[False, True])

    save_path = os.path.join(output_dir, "eda_missingness_summary.csv")
    missing_df.to_csv(save_path, index=False)
    print("Saved:", save_path)
    return missing_df


def create_basic_summary_table(df_input: pd.DataFrame, output_dir: str) -> pd.DataFrame:
    """
    Create a basic summary table for all variables.

    Parameters
    ----------
    df_input : pd.DataFrame
        Input dataframe.
    output_dir : str
        Output directory.

    Returns
    -------
    pd.DataFrame
        Basic summary table.
    """
    rows = []
    for col in df_input.columns:
        if pd.api.types.is_numeric_dtype(df_input[col]):
            s = pd.to_numeric(df_input[col], errors="coerce")
            rows.append({
                "variable": col,
                "type": "numeric",
                "n_nonmissing": int(s.notna().sum()),
                "n_missing": int(s.isna().sum()),
                "mean": round(float(s.mean()), 4) if s.notna().sum() > 0 else np.nan,
                "sd": round(float(s.std(ddof=1)), 4) if s.notna().sum() > 1 else np.nan,
                "median": round(float(s.median()), 4) if s.notna().sum() > 0 else np.nan,
                "q1": round(float(s.quantile(0.25)), 4) if s.notna().sum() > 0 else np.nan,
                "q3": round(float(s.quantile(0.75)), 4) if s.notna().sum() > 0 else np.nan,
                "min": round(float(s.min()), 4) if s.notna().sum() > 0 else np.nan,
                "max": round(float(s.max()), 4) if s.notna().sum() > 0 else np.nan,
                "n_unique": int(s.nunique(dropna=True)),
            })
        else:
            s = df_input[col].astype("object")
            top_value = s.dropna().mode()
            rows.append({
                "variable": col,
                "type": "categorical",
                "n_nonmissing": int(s.notna().sum()),
                "n_missing": int(s.isna().sum()),
                "mean": np.nan,
                "sd": np.nan,
                "median": np.nan,
                "q1": np.nan,
                "q3": np.nan,
                "min": np.nan,
                "max": np.nan,
                "n_unique": int(s.nunique(dropna=True)),
                "most_frequent": str(top_value.iloc[0]) if len(top_value) > 0 else np.nan,
            })

    summary_df = pd.DataFrame(rows)
    save_path = os.path.join(output_dir, "eda_basic_summary.csv")
    summary_df.to_csv(save_path, index=False)
    print("Saved:", save_path)
    return summary_df


def plot_eda_figures(df_input: pd.DataFrame, output_dir: str) -> None:
    """
    Generate basic EDA figures.

    Figures
    -------
    - Target distribution bar plot
    - Histograms for age, avg_glucose_level, bmi
    - Boxplots of continuous variables by stroke status
    - Missingness bar plot
    """
    os.makedirs(output_dir, exist_ok=True)

    # Target distribution
    plt.figure(figsize=(6, 4))
    df_input["stroke"].value_counts().sort_index().plot(kind="bar")
    plt.xticks([0, 1], ["No stroke", "Stroke"], rotation=0)
    plt.ylabel("Count")
    plt.title("Stroke class distribution")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "eda_target_distribution.png"), bbox_inches="tight")
    plt.close()

    # Histograms
    for col in ["age", "avg_glucose_level", "bmi"]:
        plt.figure(figsize=(6, 4))
        pd.to_numeric(df_input[col], errors="coerce").dropna().hist(bins=30)
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.title(f"Distribution of {col}")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"eda_hist_{col}.png"), bbox_inches="tight")
        plt.close()

    # Boxplots by stroke status
    for col in ["age", "avg_glucose_level", "bmi"]:
        temp = df_input[[col, "stroke"]].copy()
        temp[col] = pd.to_numeric(temp[col], errors="coerce")
        temp = temp.dropna()
        data0 = temp.loc[temp["stroke"] == 0, col].values
        data1 = temp.loc[temp["stroke"] == 1, col].values
        plt.figure(figsize=(6, 4))
        plt.boxplot([data0, data1], labels=["No stroke", "Stroke"])
        plt.ylabel(col)
        plt.title(f"{col} by stroke status")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"eda_boxplot_{col}_by_stroke.png"), bbox_inches="tight")
        plt.close()

    # Missingness plot
    missing_pct = df_input.isna().mean().sort_values(ascending=False) * 100
    plt.figure(figsize=(8, 4))
    plt.bar(missing_pct.index, missing_pct.values)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("% missing")
    plt.title("Missingness by variable")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "eda_missingness_barplot.png"), bbox_inches="tight")
    plt.close()

    print("Saved EDA figures to:", output_dir)


def create_table1(df_input: pd.DataFrame, output_dir: str) -> pd.DataFrame:
    """
    Create Table 1 comparing baseline characteristics by stroke status.

    Purpose
    -------
    Generate publication-style baseline characteristics with:
    - Overall
    - No stroke
    - Stroke
    - P value

    Statistical tests
    -----------------
    - Continuous variables: Welch's t-test
    - Categorical variables: Chi-square test

    Parameters
    ----------
    df_input : pd.DataFrame
        Original input dataframe including the target column 'stroke'.
    output_dir : str
        Output directory where CSV files will be saved.

    Returns
    -------
    pd.DataFrame
        Final Table 1 dataframe.
    """
    from scipy.stats import ttest_ind, chi2_contingency

    df_t1 = df_input.copy()
    df_t1["gender"] = df_t1["gender"].replace({"Other": "Female"})

    outcome = "stroke"
    group0 = df_t1[df_t1[outcome] == 0]
    group1 = df_t1[df_t1[outcome] == 1]

    continuous_vars = ["age", "avg_glucose_level", "bmi"]
    categorical_vars = [
        "gender", "hypertension", "heart_disease", "ever_married",
        "work_type", "Residence_type", "smoking_status"
    ]

    rows = []

    # Continuous variables
    for var in continuous_vars:
        overall_mean_sd, overall_median_iqr = summarize_continuous(df_t1[var])
        g0_mean_sd, g0_median_iqr = summarize_continuous(group0[var])
        g1_mean_sd, g1_median_iqr = summarize_continuous(group1[var])

        x0 = pd.to_numeric(group0[var], errors="coerce").dropna()
        x1 = pd.to_numeric(group1[var], errors="coerce").dropna()
        try:
            _, p_val = ttest_ind(x0, x1, equal_var=False, nan_policy="omit")
        except Exception:
            p_val = np.nan

        p_str = "<0.001" if pd.notna(p_val) and p_val < 0.001 else (f"{p_val:.3f}" if pd.notna(p_val) else "")

        rows.append({
            "Variable": var,
            "Overall": overall_mean_sd,
            "No stroke": g0_mean_sd,
            "Stroke": g1_mean_sd,
            "P value": p_str,
        })
        rows.append({
            "Variable": "  Median [IQR]",
            "Overall": overall_median_iqr,
            "No stroke": g0_median_iqr,
            "Stroke": g1_median_iqr,
            "P value": "",
        })

        n_missing = df_t1[var].isna().sum()
        if n_missing > 0:
            rows.append({
                "Variable": "  Missing",
                "Overall": f"{n_missing} ({100*n_missing/len(df_t1):.1f}%)",
                "No stroke": f"{group0[var].isna().sum()} ({100*group0[var].isna().sum()/len(group0):.1f}%)",
                "Stroke": f"{group1[var].isna().sum()} ({100*group1[var].isna().sum()/len(group1):.1f}%)",
                "P value": "",
            })

    # Categorical variables
    for var in categorical_vars:
        rows.append({
            "Variable": var,
            "Overall": "",
            "No stroke": "",
            "Stroke": "",
            "P value": "",
        })

        contingency = pd.crosstab(df_t1[var], df_t1[outcome])
        try:
            _, p_val, _, _ = chi2_contingency(contingency)
        except Exception:
            p_val = np.nan
        p_str = "<0.001" if pd.notna(p_val) and p_val < 0.001 else (f"{p_val:.3f}" if pd.notna(p_val) else "")

        categories = df_t1[var].dropna().astype(str).unique().tolist()
        categories = sorted(categories)
        first = True
        for cat in categories:
            n_all = (df_t1[var].astype(str) == cat).sum()
            n_0 = (group0[var].astype(str) == cat).sum()
            n_1 = (group1[var].astype(str) == cat).sum()

            rows.append({
                "Variable": f"  {cat}",
                "Overall": f"{n_all} ({100*n_all/len(df_t1):.1f}%)",
                "No stroke": f"{n_0} ({100*n_0/len(group0):.1f}%)",
                "Stroke": f"{n_1} ({100*n_1/len(group1):.1f}%)",
                "P value": p_str if first else "",
            })
            first = False

    table1_df = pd.DataFrame(rows)
    table1_path = os.path.join(output_dir, "table1_baseline_characteristics.csv")
    table1_df.to_csv(table1_path, index=False)

    print("Saved:", table1_path)
    return table1_df


# Run EDA and Table 1 generation using the original dataset before modeling
try:
    print("\nRunning EDA summaries and Table 1 generation...")
    missing_df = create_missingness_table(df_input=df, output_dir=OUTPUT_DIR)
    basic_summary_df = create_basic_summary_table(df_input=df, output_dir=OUTPUT_DIR)
    plot_eda_figures(df_input=df, output_dir=OUTPUT_DIR)
    table1_df = create_table1(df_input=df, output_dir=OUTPUT_DIR)

    print("\nEDA missingness summary preview:")
    print(missing_df.head(10))
    print("\nEDA basic summary preview:")
    print(basic_summary_df.head(10))
    print("\nTable 1 preview:")
    print(table1_df.head(20))
except Exception as e:
    print(f"EDA/Table 1 generation failed: {e}")


# ============================================
# 11. Optional: simple interpretation notes
# ============================================
print("\nInterpretation notes:")
print("1) This pipeline prioritizes sensitivity to minimize false negatives in a screening-like clinical setting.")
print("2) AUROC alone is insufficient in highly imbalanced data; PR-AUC and calibration are also required.")
print("3) External validation is mandatory before clinical deployment.")
print("4) For publication-grade rigor, nested CV, MI pooling, and decision-curve analysis are recommended.")


Please choose the stroke CSV file from your local computer.


Saving 3.stroke.csv to 3.stroke.csv
Uploaded file detected: 3.stroke.csv
Dataset shape: (5110, 12)
Positive class proportion: 0.0487279843444227
Missing values:
 id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64
scale_pos_weight for XGBoost: 19.522
Completed: Logistic Regression | none
Completed: Logistic Regression | class_weight
Completed: Logistic Regression | smote
Completed: Decision Tree | none
Completed: Decision Tree | class_weight
Completed: Decision Tree | smote
Completed: Random Forest | none
Completed: Random Forest | class_weight
Completed: Random Forest | smote
Completed: Extra Trees | none
Completed: Extra Trees | class_weight
Completed: Extra Trees | smote
Completed: MLP | none
Completed: MLP | class_weight
Comp